# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

> **Note:** All dataset elements—record set, fields, columns, etc.—are referenced by their `@id` fields as per the FAIR2 schema.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Get metadata object and print title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Dataset ID (@id): {dataset.metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we query the available record sets and enumerate their fields, referencing everything by `@id` as required.

In [ ]:
# List all record sets with their @ids
record_sets = dataset.metadata.record_sets
print(f"Number of RecordSets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    print("Fields:")
    for field in rs.fields:
        print(f"  - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print("-----\n")

## 3. Data Extraction
Load data from specific record sets into Pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

**Example:** Load each record set referenced by `@id`.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records for each record set and create DataFrame
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for record_set @id '{record_set_id}':\n  Columns: {list(df.columns)}\n  Preview:\n{df.head(2)}\n")

# Choose a record set for downstream analysis (e.g., first one)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Proceeding with record set @id: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping data by attributes.

Reference all fields by their `@id`. Adjust the IDs below if they differ in your actual overview (see above cell).

In [ ]:
# Choose numeric and group field @id from main record set
if main_record_set_id:
    main_df = dataframes[main_record_set_id]
    # Print available column @ids
    print("Available column @ids:", list(main_df.columns))

    # Example: Choose log likelihood field and ward/group field (replace the @ids below appropriately)
    numeric_field_id = None
    group_field_id = None
    
    # Attempt to select fields by common @id or column name
    for col in main_df.columns:
        if 'log_likelihood' in col.lower() or 'loglikelihood' in col.lower():
            numeric_field_id = col
        elif 'ward' in col.lower() or 'location' in col.lower():
            group_field_id = col

    if numeric_field_id:
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:\n{filtered_df.head()}\n")

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:\n{filtered_df[[numeric_field_id, norm_col]].head()}\n")

        # Grouping if group field exists
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:\n{grouped_df.head()}\n")
    else:
        print("No numeric field @id found in main record set. Please refer to the overview above.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using the columns referenced by their `@id`.

In [ ]:
# Example visualization: distribution/histogram and group comparison
if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    filtered_df[numeric_field_id].hist(bins=20)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f'Histogram of {numeric_field_id} (filtered)')
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().plot(kind='bar')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset covers ordered logistic regression results for household adoption predictors in rangeland management.
- We loaded metadata and explored multiple record sets using their `@id`s as defined in the Croissant schema.
- Example EDA shows log likelihood distributions and grouping by ward/location (if present).
- The dataset enables deeper analysis for policy, research, and community interventions in pastoralist settings.

> For further analysis, reference all fields/columns explicitly by their `@id` as per the FAIR^2 schema.